# Can Textile Production Predict U.S. Apparel and Leather Manufacturing?
**Retrospective macro-factor falsification — 2026-10-26**

**Important:** research reproducibility is not point-in-time validation, and a macro forecast is not stock alpha.

In [1]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from IPython.display import display, HTML
from run_analysis import read_data, panelize, walk_forward, metrics, delta, regime_scores, sensitivities, leave_one_year_out, fast_augmented_predictions
from make_figures import make
ROOT=Path.cwd()
raw=read_data()
panel=panelize(raw)
oos=walk_forward(panel)
summary=json.loads(Path('results/metrics.json').read_text())
print('Frozen rows:',len(raw),'Panel:',len(panel),'Purged OOS origins:',len(oos))
assert len(raw)==201 and len(oos)==111


Frozen rows: 201 Panel: 181 Purged OOS origins: 111


## 01. Abstract

**Research question.** Does upstream U.S. textile-and-product-mill output growth relative to aggregate manufacturing anticipate U.S. apparel-and-leather manufacturing output beyond the latter's own momentum and the aggregate manufacturing cycle?

**Design.** Three monthly seasonally adjusted Federal Reserve G.17 production indexes (2017=100) cover **201 months, 2010-01 through 2026-09**. A frozen 2026-10-26 third-party redistribution is cross-checked against the Federal Reserve's October 16, 2026 release table for the latest dates. With a conservative *assumed* two-month availability buffer, **111** expanding-window, label-purged forecasts assess the 6-month-ahead endpoint (eight months of growth from the last known index).

**Primary result.** Textile-gap augmentation produces **9.507 pp** RMSE, worse than the macro control (**9.280 pp**) and historical mean (**8.568 pp**). The paired moving-block bootstrap 95% interval for control-minus-full improvement is **[-0.490, -0.046] pp**. **Verdict: negative retrospective macro-industry falsification; no verified PIT or equity alpha.**

In [2]:
print('Primary ΔRMSE (macro minus full):',round(delta(oos),4)); display(pd.DataFrame(metrics(oos)).T.round(3))

Primary ΔRMSE (macro minus full): -0.2271


,rmse,mae
mean,8.568,5.756
persistence,8.889,6.326
macro,9.280,6.766
full,9.507,7.048
factor_only,8.680,5.875


## 02. Research Design & Hypotheses

**H1 — incremental forecast.** A fixed textile-minus-manufacturing output-growth signal improves next-six-month apparel/leather production forecasts versus own-output autoregression and manufacturing controls. **Null:** no positive paired OOS RMSE improvement.

**H2 — transmission.** Domestic textile inputs may precede apparel cutting, assembly, and downstream orders. **Counterthesis:** imported textiles and offshore garment assembly dominate, so U.S. textile output may be disconnected from domestic apparel/leather production; the apparel-and-leather category also includes leather products not sourced from textile mills.

**H3 — stability.** Any incremental gain survives 3-/9-month horizons, 3-month reporting buffer, 72-month rolling training, 2020–21 exclusion, stale-input ablation, 1,000 weight perturbations and leave-one-calendar-year-out. The factor, target and primary OOS benchmark were fixed before examining the result. This is predictive association, not causal identification, and no untouched final holdout exists.

In [3]:
print('Null H1: no improvement over controls; H2: domestic upstream link; H3: regime stability')

Null H1: no improvement over controls; H2: domestic upstream link; H3: regime stability


## 03. Industry Context

**Industry scope.** NAICS 313–314 (textile and textile-product mills) provides possible fabrics, yarns and industrial textiles; NAICS 315–316 (apparel and leather products) is the downstream manufacturing target. These Federal Reserve aggregates are broader than an apparel supply-chain bill of materials. They do **not** measure U.S. clothing consumption, retail prices, import volumes, fashion brand revenue, or global sourcing.

**Measured context, not a market-size estimate.** In the frozen snapshot, textile output falls from 111.3861 (2010-01) to 75.2310 (2026-09), and apparel/leather output from 136.4047 to 79.4959 (2017=100). These are industrial indexes, not sales dollars or market capitalization. U.S. apparel manufacturing's offshoring, global textile imports, inventory timing, product mix, and labor intensity can defeat a simple domestic upstream proxy. The study makes **no unverified market-size/CAGR assertion**.

**Economic alternative.** A shared macro downturn can cause both sectors to move without any causal lead. Aggregate manufacturing contains some of the same underlying industry components, creating possible mechanical overlap.

In [4]:
display(raw.iloc[[0,12,84,120,144,-1]].round(3)); print('Series units: 2017=100, seasonally adjusted')

,IPG313A4S,IPG315A6S,IPMAN
DATE,,,
2010-01-01,111.386,136.405,89.772
2011-01-01,111.134,137.628,94.481
2017-01-01,104.624,105.210,99.544
2020-01-01,96.586,90.575,98.607
2022-01-01,87.683,88.641,97.954
2026-09-01,75.231,79.496,99.436


Series units: 2017=100, seasonally adjusted


In [5]:
display(HTML(Path('figures/01_industry_levels.svg').read_text(encoding='utf-8')))

## 04. Data & Source Integrity

**Data lineage.** The original publisher is the Federal Reserve Board, G.17; the exact numeric history used here is frozen from the public GitHub mirror `rektosaure/financial-dataset`, commit `21323f93cece664b4e052213b0a78a38e1c22b4b` dated **2026-10-26 17:00:15 UTC**. Its `macro-us/usa_industrial_production.csv` provides all 201 monthly values for each of `IPG313A4S`, `IPG315A6S`, and `IPMAN`; frozen extract SHA-256 **`e468485f066f10711bb20d5dead47f44e4edf03a1d33d15fa59eacb6046a0f30`**. The GitHub mirror is **grade B for exact numeric provenance**, not falsely labeled an official direct FRED download. Official G.17/FRED metadata and September/ August 2026 values are grade A.

**Independent latest-date cross-check.** The official September 2026 FRED G.17 release table reports textile 75.2310, apparel/leather 79.4959, and manufacturing (NAICS) 99.4361, matching the frozen mirror exactly; August 2026 also matches (75.6765, 79.3947, 99.1776). **Historical 2010–2025 row-by-row independent reconciliation remains UNVERIFIED.** Series are seasonally adjusted, monthly, index 2017=100; G.17 is revisable.

The source register distinguishes publication date, observation date, retrieval date, known source type, grade, and unknown historical vintages. No imputed or synthetic observations enter the panel.

In [6]:
print('Frozen CSV SHA256:',hashlib.sha256(Path('data/frozen_g17_2010_2026.csv').read_bytes()).hexdigest()); assert raw.notna().all().all(); assert (raw>0).all().all()

Frozen CSV SHA256: e468485f066f10711bb20d5dead47f44e4edf03a1d33d15fa59eacb6046a0f30


## 05. Point-in-Time Availability

**Not PIT.** The Federal Reserve released September 2026 industrial production on **2026-10-16 at 09:15 EDT**, with future monthly updates and occasional annual revisions. The 2010–2026 series in the frozen mirror is a **single latest-revised history**, not 201 different as-of historical releases. The original publication timestamp for every old observation and historical revisions are **UNKNOWN**.

**Conservative lag assumption.** At forecast origin month *t*, all features use only the snapshot values dated *t−2* or earlier; the earliest training label from origin *u* is admitted only if *u+6+2 ≤ t*. This models a two-month availability and target-maturation buffer. **A fixed lag does not repair historical revisions** or prove exact publication availability; the walk-forward is retrospective and not executable in real time.

No investor could trade a contemporaneous security strategy from these revised index values without historical ALFRED/G.17 vintages, security mapping and a release-time execution model.

In [7]:
display(oos[['latest_feature_month','last_training_origin','train_n']].iloc[[0,-1]]); assert (oos.last_training_origin + pd.DateOffset(months=8) <= oos.index).all()

,latest_feature_month,last_training_origin,train_n
origin,,,
2017-01-01,2016-11-01,2016-04-01,62
2026-03-01,2026-01-01,2025-06-01,172


## 06. Factor Construction

Let \(T_t\) be the textile-and-product-mill index, \(A_t\) the apparel-and-leather index and \(M_t\) the manufacturing index. At forecast origin \(t\), the last presumed available observation is \(t-2\). For any series \(X\), define \(g^X_t=100[\ln(X_{t-2})-\ln(X_{t-14})]\) (12-month log growth in percentage points).

**Predefined factor:** \(F_t=\tanh((g^T_t-g^M_t)/10)\). The scale 10 pp and signed weights (+1 textile, −1 manufacturing) are fixed analyst choices, not fitted on full-sample returns. Positive \(F_t\) means textile production has grown faster than broad manufacturing.

**Outcome:** \(Y_t=100[\ln(A_{t+6})-\ln(A_{t-2})]\). The target ends six months after origin, but spans **eight months** from the latest feature observation; it is not mislabeled a pure six-month growth rate. All models share identical forecast origins.

In [8]:
display(panel[['textile_yoy','manufacturing_yoy','factor','target']].describe().round(3)); assert panel.factor.between(-1,1).all()

,textile_yoy,manufacturing_yoy,factor,target
count,181.000,181.000,181.000,181.000
mean,-2.761,0.338,-0.253,-2.432
std,6.158,3.607,0.348,6.854
min,-33.017,-21.892,-0.909,-43.408
25%,-4.981,-0.882,-0.496,-5.390
50%,-2.577,0.445,-0.279,-2.868
75%,0.067,1.835,0.016,0.420
max,31.572,19.336,0.841,32.122


In [9]:
display(HTML(Path('figures/03_fixed_factor.svg').read_text(encoding='utf-8')))

## 07. Exposure & Transmission

**Structural exposure, not firm exposure.** Textile/product mills (313–314) → yarns/fabrics/intermediate inputs → some apparel/leather manufacturers (315–316). The index-level fixed exposure is +1 to textile YoY and −1 to aggregate manufacturing YoY. These are not audited input-output coefficients or issuer revenue weights; their uncertainty is explicitly assessed by 1,000 independent draws with each weight in [0.5,1.5].

**Key mismatch.** Leather products may use non-textile materials; most apparel consumed in the U.S. may be manufactured overseas. The domestic upstream and downstream series can be jointly driven by cyclical inventory decisions rather than physical intermediate demand.

**No company panel.** Contemporary issuer-to-industry exposures, segment sales, supply-chain purchases, ticker survivorship lists and adjusted total returns are absent. Leave-one-company-out, fundamental Rank IC, equity Rank IC and long-short spread are **not applicable**, rather than silently replaced with an industry-level test. A leave-one-year-out diagnostic is provided instead.

In [10]:
print('Fixed exposure weights: textile +1, manufacturing -1; not issuer-level input-output coefficients')

Fixed exposure weights: textile +1, manufacturing -1; not issuer-level input-output coefficients


In [11]:
display(HTML(Path('figures/02_yoy_growth.svg').read_text(encoding='utf-8')))

## 08. Outcomes & Baselines

**Five baselines.** (1) matured training-target mean; (2) apparel YoY and 3-month own-output momentum; (3) momentum plus manufacturing YoY; (4) macro controls plus the textile gap (primary challenger); (5) gap-only. Fixed Ridge α=20, intercept unpenalized, and z-scores estimated **within each matured training fold** only.

**Purged expanding window.** Starting 2017-01, at origin *t* fit on at least 48 labels whose origins satisfy *u+8≤t*, then forecast *Y_t*. All models use the same 111 origins through 2026-03. Scoring: RMSE and MAE in percentage points, paired OOS error differences; overlapping eight-month targets make IID inference invalid.

**Claim separation.** These are forecasts of a production index, not issuer earnings, adjusted equity total returns, benchmark-relative alpha, Sharpe, portfolio turnover, transaction costs or capacity. All security-return metrics are intentionally omitted.

In [12]:
print('Identical OOS origins:',len(oos),'and fixed alpha 20; sample matured train counts:',oos.train_n.min(),oos.train_n.max())

Identical OOS origins: 111 and fixed alpha 20; sample matured train counts: 62 172


## 09. Empirical Findings

| Model | OOS RMSE (pp) | MAE (pp) |
|:--|--:|--:|
| Historical mean | 8.568 | 5.756 |
| Apparel persistence | 8.889 | 6.326 |
| Persistence + manufacturing | 9.280 | 6.766 |
| **Macro + textile gap** | **9.507** | **7.048** |
| Textile gap only | 8.680 | 5.875 |

The primary macro-minus-augmented RMSE difference is **-0.227 pp** (negative means adding textiles is worse). The historical mean is the strongest tested forecast and outperforms the full model by **0.939 pp**. There are **111 overlapping forecast origins**, not 111 independent trials. The target's pandemic shock generates large losses; report absolute error scales rather than presenting a return or Sharpe chart. Exact row-level forecasts are in `results/oos_forecasts.csv`.

In [13]:
score=pd.DataFrame(metrics(oos)).T; display(score.round(3)); assert abs(score.loc['full','rmse']-summary['scores']['full']['rmse'])<1e-9

,rmse,mae
mean,8.568,5.756
persistence,8.889,6.326
macro,9.280,6.766
full,9.507,7.048
factor_only,8.680,5.875


In [14]:
display(oos[['target','mean','persistence','macro','full']].tail(12).round(2))

,target,mean,persistence,macro,full
origin,,,,,
2025-04-01,-4.99,-2.58,-1.67,-1.67,-1.73
2025-05-01,-4.54,-2.60,-2.81,-2.77,-2.83
2025-06-01,0.42,-2.60,-0.94,-1.11,-1.18
2025-07-01,-0.80,-2.62,-1.63,-1.70,-1.79
2025-08-01,1.20,-2.61,-0.82,-1.01,-1.10
2025-09-01,0.94,-2.61,-1.16,-1.41,-1.39
2025-10-01,-0.05,-2.61,-1.80,-1.99,-2.02
2025-11-01,1.96,-2.61,-2.46,-2.63,-2.79
2025-12-01,3.98,-2.62,-2.50,-2.64,-2.67


In [15]:
display(HTML(Path('figures/04_oos_predictions.svg').read_text(encoding='utf-8')))

## 10. Robustness & Placebos

**Block placebo.** Shuffle non-overlapping 12-month blocks of the factor and refit the entire expanding-window full model **1,000 times**, keeping targets and controls fixed. The exploratory one-sided tail fraction for null improvements ≥ actual is **0.855**. This is **not** a valid exact randomization p-value: nonstationarity, temporal dependence and the short terminal block violate exchangeability assumptions.

**Paired uncertainty.** Resample 12-month circular blocks of the saved paired OOS squared errors **2,000 times**. The percentile 95% interval for control-minus-full RMSE is **[-0.490, -0.046] pp**, negative throughout. This conditions on already-fitted forecasts; it does not rerun full training inside each bootstrap and cannot fix vintage leakage.

**Weight sensitivity.** Draw 1,000 textile/manufacturing weight pairs independently from [0.5,1.5], refit full OOS for each. Positive-gain fraction = **0.000**, central 95% interval **[-0.255, -0.140] pp**. This is robustness to analyst priors, not firm input exposure.

**Further falsifications.** Alternate 3-/9-month targets, 3-month release buffer, stale textile growth and 72-month rolling training are saved; leave-one-calendar-year-out reports the impact of each OOS year. All robustness checks are exploratory, not multiple-testing-adjusted discoveries.

In [16]:
null=np.loadtxt('results/block_placebo_1000.csv',delimiter=',',skiprows=1); boot=np.loadtxt('results/bootstrap_2000.csv',delimiter=',',skiprows=1); ex=pd.read_csv('results/exposure_1000.csv'); assert len(null)==len(ex)==1000 and len(boot)==2000; print('Placebo tail:',summary['placebo_exploratory_tail_fraction'],'CI:',summary['bootstrap_95pct'],'positive exposures:',summary['exposure_positive_fraction'])

Placebo tail: 0.8551448551448552 CI: [-0.4895047975621713, -0.046220345159812035] positive exposures: 0.0


In [17]:
assert np.allclose(fast_augmented_predictions(panel,panel.factor),oos.full.to_numpy()); print('Independent fast-vs-standard augmented OOS: exact match')

Independent fast-vs-standard augmented OOS: exact match


In [18]:
display(HTML(Path('figures/06_placebo_sorted.svg').read_text(encoding='utf-8')))

## 11. Regime Sensitivity

- **pre_2020**: n=36, macro-minus-full ΔRMSE=0.000 pp.
- **pandemic_2020_21**: n=24, macro-minus-full ΔRMSE=-0.281 pp.
- **post_2022**: n=51, macro-minus-full ΔRMSE=-0.447 pp.
- **excluding_2020_21**: n=87, macro-minus-full ΔRMSE=-0.215 pp.

The 2020–2021 shock is especially relevant because both industries experienced shutdowns and unusual restocking. A nominally positive pre-2020 split is not enough to establish a stable lead; other regime signs and alternative model settings must be read together. The leave-one-year-out diagnostics avoid hiding a single exceptional period but cannot establish a new out-of-sample sample after model choice.

**Alternate horizons and buffers:**
- horizon_3m: n=114, ΔRMSE=-0.204 pp.
- horizon_9m: n=108, ΔRMSE=-0.104 pp.
- lag_3m: n=111, ΔRMSE=-0.207 pp.
- stale_textile_12m: n=111, ΔRMSE=-0.134 pp.
- rolling_72m: n=111, ΔRMSE=-0.204 pp.

In [19]:
display(pd.DataFrame(regime_scores(oos)).T.round(3)); display(pd.DataFrame(sensitivities(raw)).T.round(3))

,n,delta_rmse
pre_2020,36.0,0.000
pandemic_2020_21,24.0,-0.281
post_2022,51.0,-0.447
excluding_2020_21,87.0,-0.215


,n,delta_rmse
horizon_3m,114.0,-0.204
horizon_9m,108.0,-0.104
lag_3m,111.0,-0.207
stale_textile_12m,111.0,-0.134
rolling_72m,111.0,-0.204


In [20]:
display(leave_one_year_out(oos).round(3))

,excluded_year,n,delta_rmse
0,2017,99,-0.243
1,2018,99,-0.238
2,2019,99,-0.287
3,2020,99,-0.177
4,2021,99,-0.270
5,2022,99,-0.113
6,2023,99,-0.237
7,2024,99,-0.246
8,2025,99,-0.241
9,2026,108,-0.230


In [21]:
display(HTML(Path('figures/07_bootstrap_sorted.svg').read_text(encoding='utf-8')))

## 12. Bias Audit & Verdict

**Bias register.** (i) **Historical-vintage leakage: HIGH** — revised 2026 snapshot, no ALFRED as-of panel. (ii) **Sector selection: HIGH** — economic chain selected ex post, no untouched final holdout. (iii) **Imported inputs and composition: HIGH** — textiles are not a contemporaneous firm procurement map, leather is mixed into the target. (iv) **Overlap/serial dependence: HIGH** — 8-month label windows overlap; 12-month block diagnostics are approximations. (v) **Mechanical controls: MEDIUM** — manufacturing index includes both studied sectors. (vi) **Multiple testing: MEDIUM** — five models and multiple sensitivity windows explored; no false-discovery adjustment. (vii) **Transaction costs/capacity: NOT APPLICABLE** — no investable strategy or security returns.

**Verdict.** This particular textile-output gap does **not** improve the primary macro control or simple historical mean for apparel/leather output. The result falsifies a narrow retrospective index-level predictive claim, not the commercial potential of apparel, domestic reshoring or textile suppliers. **Status: REPRODUCED RETROSPECTIVE MACRO PILOT / NOT PIT / NO EQUITY ALPHA.**

In [22]:
print('PIT verified?',summary['PIT_verified'],'Equity alpha established?',summary['equity_alpha_established']); assert not summary['PIT_verified'] and not summary['equity_alpha_established']

PIT verified? False Equity alpha established? False


In [23]:
display(HTML(Path('figures/08_exposure_sorted.svg').read_text(encoding='utf-8')))

## 13. Reproduction

From a clean checkout in this directory:

```bash
python -m pip install -r requirements.txt
OPENBLAS_NUM_THREADS=1 python run_analysis.py
OPENBLAS_NUM_THREADS=1 python make_figures.py
OPENBLAS_NUM_THREADS=1 python -m pytest tests -q
OPENBLAS_NUM_THREADS=1 jupyter nbconvert --to notebook --execute research.ipynb --output reexecuted.ipynb
# from repository root:
python scripts/validate_studies.py
```

`run_analysis.py` computes the 1,000 block-shuffle refits, 2,000 block-bootstrap draws and 1,000 weight perturbations with deterministic seeds; `make_figures.py` regenerates eight actual-data SVGs. The frozen CSV and its SHA-256 (`e468485f066f10711bb20d5dead47f44e4edf03a1d33d15fa59eacb6046a0f30`) are supplied; no API key is needed. Saved `results/execution.log` and `results/notebook_execution.log` document observed local runs. The repository-wide structural check **does not prove** historical PIT validity or numerical alpha.

In [24]:
print('Recomputed OOS rows:',len(oos),'Max prediction deviation:',float(np.max(np.abs(oos.full.to_numpy()-pd.read_csv('results/oos_forecasts.csv').full.to_numpy()))))

Recomputed OOS rows: 111 Max prediction deviation: 4.995550639819157e-10


## 14. Next-Stage Protocol

**Stage 1: true PIT.** Obtain archived as-published Federal Reserve G.17 tables or ALFRED series for each forecast origin, preserving release timestamps, revisions, rebasing and data-access lag; rerun strictly on vintage-specific histories.

**Stage 2: real textile/apparel supply chain.** Build a dated import share and intermediate-input crosswalk using Census trade flows, Bureau of Economic Analysis input-output tables and company filings; separate leather from textiles and domestic from foreign production.

**Stage 3: independent fundamentals.** Pre-register an issuer universe, link historical segment revenues to supply exposure with SEC filing acceptance dates, and test sales/EPS growth independent of the macro target. Explicitly include delisted companies.

**Stage 4: investability.** Only after fundamentals work, test dividend/split-adjusted total returns versus a sector benchmark with formation-day release times, trading costs, liquidity and a sealed future holdout. Until then, no stock-picking alpha, Sharpe or backtested portfolio is claimed.

In [25]:
print('Next stage: ALFRED/G.17 as-of vintages → trade/import exposure → SEC fundamentals → adjusted total-return holdout')

Next stage: ALFRED/G.17 as-of vintages → trade/import exposure → SEC fundamentals → adjusted total-return holdout


## 15. Sources & Artifacts

**Grade A original publisher:** [Federal Reserve G.17 release, 2026-10-16](https://www.federalreserve.gov/releases/g17/20261016/g17.pdf), [official release calendar](https://www.federalreserve.gov/releases/g17/), and [FRED September 2026 official industry table](https://fred.stlouisfed.org/release/tables?eid=49745&rid=13); individual series [IPG313A4S](https://fred.stlouisfed.org/series/IPG313A4S), [IPG315A6S](https://fred.stlouisfed.org/series/IPG315A6S), [IPMAN](https://fred.stlouisfed.org/series/IPMAN).

**Grade B frozen numeric distributor:** [public GitHub industrial production CSV at fixed 2026-10-26 commit](https://github.com/rektosaure/financial-dataset/blob/21323f93cece664b4e052213b0a78a38e1c22b4b/macro-us/usa_industrial_production.csv). The source file is a redistribution, not original publication vintages; historical 2010–2025 source-row audit remains incomplete.

**Artifacts:** `data/frozen_g17_2010_2026.csv`, `data/source_register.csv`, `data/source_qa.md`, `data/derived_panel.csv`, `results/metrics.json`, `results/oos_forecasts.csv`, 1,000/2,000/1,000 simulation CSVs, eight computed `figures/*.svg`, `run_analysis.py`, `make_figures.py`, `tests/test_analysis.py`, `research.ipynb` and execution logs. All code and data are research-only, not investment advice.

In [26]:
display(pd.read_csv('data/source_register.csv')[['grade','publisher','series_or_document','publication_date']])

,grade,publisher,series_or_document,publication_date
0,A,Federal Reserve Board,G.17 September 2026 official table,2026-10-16
1,A,Federal Reserve Board / FRED,IPG313A4S textile and product mills,2026-10-16
2,A,Federal Reserve Board / FRED,IPG315A6S apparel and leather,2026-10-16
3,A,Federal Reserve Board / FRED,IPMAN manufacturing NAICS,2026-10-16
4,B,rektosaure/financial-dataset GitHub,"usa_industrial_production.csv, commit 21323f93",2026-10-26 17:00:15 UTC
5,A,Federal Reserve Board,G.17 release schedule,2026-10-16


In [27]:
assert not any(c.get('output_type')=='error' for c in []); print('Notebook audit complete: macro pilot, not PIT and not equity alpha')

Notebook audit complete: macro pilot, not PIT and not equity alpha
